# 🗓️ Timeline Generator — MIMP
**Ejecuta todo con `Ctrl+F9` y sube tu archivo cuando te lo pida.**

Repositorio: [https://github.com/KEVINmarce1996/timeline-generator-mimp](https://github.com/KEVINmarce1996/timeline-generator-mimp)

---

In [ ]:
# CELDA 1 — Instalación automática (no modificar)
import subprocess, sys, shutil, zipfile, urllib.request
from pathlib import Path

# Instalar dependencias
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "openpyxl", "python-pptx", "python-docx", "pdfplumber",
    "dateparser", "google-genai", "Pillow", "lxml"], check=True)

# Limpiar instalación anterior
proj_dir = Path("/content/timeline_generator")
if proj_dir.exists():
    shutil.rmtree(proj_dir)
to_remove = [k for k in sys.modules if k.startswith(("core","utils","gui"))]
for k in to_remove: del sys.modules[k]

# Descargar código desde GitHub automáticamente
ZIP_URL  = "https://github.com/KEVINmarce1996/timeline-generator-mimp/raw/main/timeline_generator_v2.zip"
zip_path = Path("/content/timeline_generator_v2.zip")
print("⬇ Descargando código desde GitHub...")
urllib.request.urlretrieve(ZIP_URL, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")
sys.path.insert(0, "/content/timeline_generator")

# Verificar
ok = all((proj_dir / f).exists() for f in [
    "core/pptx_generator.py", "core/file_parser.py",
    "core/timeline_builder.py", "assets/formato_excel_modelo.xlsx"])
print(f"✓ Instalación correcta: {ok}")
if not ok:
    raise RuntimeError("❌ Error descargando el código. Intenta de nuevo.")


In [ ]:
# CELDA 2 — Cargar módulos (no modificar)
import importlib, logging
logging.basicConfig(level=logging.WARNING)

import core.model_analyzer;  importlib.reload(core.model_analyzer)
import core.file_parser;      importlib.reload(core.file_parser)
import core.timeline_builder; importlib.reload(core.timeline_builder)
import core.pptx_generator;   importlib.reload(core.pptx_generator)
import utils.date_detector;   importlib.reload(utils.date_detector)

from core.model_analyzer   import ModelAnalyzer
from core.file_parser      import FileParser
from core.timeline_builder import TimelineBuilder, BuilderConfig
from core.pptx_generator   import PptxGenerator
from utils.date_detector   import DetectorConfig
from datetime import date

template = ModelAnalyzer(
    "/content/timeline_generator/assets/formato_excel_modelo.xlsx"
).extract()

print(f"✓ Sistema listo | Fecha: {date.today()}")
print("  Hito pasado → gris | En curso → azul | Futuro → amarillo")


In [ ]:
# CELDA 3 — Sube tu archivo aquí ⬆
# Formatos válidos: .png .jpg .docx .pdf .xlsx
from google.colab import files as cf
from pathlib import Path

UPLOAD_DIR = Path("/content/mis_archivos")
UPLOAD_DIR.mkdir(exist_ok=True)

print("📂 Selecciona tu archivo de cronograma")
print("   (captura SEACE, Word, PDF o Excel)")
uploaded = cf.upload()

SUPPORTED  = {".xlsx",".xls",".docx",".pdf",".png",".jpg",".jpeg"}
INPUT_PATHS = []
for name, data in uploaded.items():
    ext = Path(name).suffix.lower()
    if ext not in SUPPORTED:
        print(f"  ⚠ Formato no soportado: {name}")
        continue
    dest = UPLOAD_DIR / name
    dest.write_bytes(data)
    INPUT_PATHS.append(dest)
    print(f"  ✓ {name}  ({len(data)//1024} KB)")

print(f"\n{len(INPUT_PATHS)} archivo(s) listo(s) para procesar")


In [ ]:
# CELDA 4 — Generar PowerPoint y descargar
from pathlib import Path
from google.colab import files as cf

USE_VISION = True  # Lee imágenes con Gemini Vision (requiere GOOGLE_API_KEY)

if not INPUT_PATHS:
    print("❌ No hay archivos. Ejecuta primero la Celda 3.")
else:
    parser  = FileParser(DetectorConfig(granularity="annual"),
                         use_vision=USE_VISION)
    builder = TimelineBuilder(BuilderConfig(granularity="annual",
                                            max_columns=14))
    LAYOUTS = []
    for filepath in INPUT_PATHS:
        print(f"\n📄 {filepath.name}")
        try:
            parsed = parser.parse(filepath)
            layout = builder.build(parsed)
            LAYOUTS.append(layout)
            print(f"   Proyecto : {layout.project_title}")
            for s in layout.sections:
                print(f"   [{s.title}]")
                for col in s.columns:
                    for e in col.events:
                        d = e.detected_date.date_start.strftime("%d/%m/%Y") \
                            if e.detected_date and e.detected_date.date_start else "?"
                        print(f"     [{e.status:7}] {d} | {e.label[:45]}")
        except Exception:
            import traceback; traceback.print_exc()

    if LAYOUTS:
        OUT = Path("/content/output"); OUT.mkdir(exist_ok=True)
        gen = PptxGenerator(template)
        for layout in LAYOUTS:
            gen.add_layout(layout)
        pptx_file = OUT / "timeline.pptx"
        gen.save(pptx_file)
        print(f"\n✅ PowerPoint generado")
        print("📥 Descargando a tu PC...")
        cf.download(str(pptx_file))
        print("\n¡Listo! Revisa tu carpeta de Descargas 🎉")
    else:
        print("❌ No se generó ningún layout. Revisa el archivo subido.")
